## Procesamiento del Resultado de la API

En esta sección se normaliza la información obtenida desde la API para transformarla en un formato consistente y dejarla estructurada de la misma manera que el *dataframe* principal.


In [47]:
from __future__ import annotations

import json
import logging
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional


PATH = './scopus_results_fieldv2.json'

In [48]:
fields = {
    'authors_info': 'Authors',
    'authors_info_with_id': 'Author full names',
    'authors_id': 'Author(s) ID',
    'dc:title': 'Title',
    'prism:coverDate': 'Date',
    'prism:volume': 'Volume',
    'prism:issueIdentifier': 'Issue',
    'citedby-count': 'Cited by',
    'prism:doi': 'DOI',
    'subtypeDescription': 'Document Type'
}

## __Necesito__

| Clave                  | Descripción (general) |
|-------------------------|-----------------------|
| `dc:title`            | Título del artículo/documento |
| `prism:publicationName`| Nombre de la revista o fuente |
| `prism:volume`        | Volumen de la publicación |
| `prism:issueIdentifier`| Número de la edición (issue) |
| `prism:pageRange`     | Páginas del artículo |
| `prism:coverDate`     | Fecha de publicación (ISO: YYYY-MM-DD) |
| `prism:doi`           | DOI del artículo |
| `citedby-count`       | Número de citas |
| `subtypeDescription`   | Descripción del subtipo (ej. Article, Review) |
| `author`               | Lista de autores |


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format = "%(levelname)s:%(name)s:%(message)s") # esto es para probar acá
logger = logging.getLogger(__name__)


class ElsevierSchemaError(RuntimeError):
    """It notes that the JSON response does not meet the contract expected from Elsevier/Scopus."""


class Elsjson: 
    """
    Parser/normalizer for Elsevier/Scopus Search Results JSON responses.

    Assumed contract (stable API):
    - The root must contain the key 'search-results' (dict).
    - 'search-results' must contain 'entry' (list of dicts). May be an empty list.
    - Each 'entry' contains metadata of the result (title, authors, etc.).

    Typical usage:
    parser = ElsevierSearchJSONParser(file_path="response.json", fields_map=FIELDS)
    df = parser.run()

    Parameters
    ----------
    file_path : str | Path | None
        Path to the local JSON file (when processing from disk).
    encoding : str
        Encoding of the JSON file (default 'utf-8').
    """

    def __init__(
        self, 
        path: Optional[str | Path] = None, 
        encoding: str = 'utf-8'
    ) -> None:
        self.path: Optional[Path] = Path(path) if path else None
        self._data: Optional[Dict[str, Any]] = None
        self.encoding: str = encoding


    @property
    def path(self):
        """ Gets the path"""
        if not self._path:
            raise AttributeError("path has not been loaded yet. Call self.path = Path(...)")
        return self._path
    
    @path.setter
    def path(self, path):
        """ Sets the path"""
        self._path = Path(path)

    @property
    def data(self):
        """Returns the parsed JSON data as a Python object (dict or list)."""
        if not self._data:
            raise AttributeError(
                "JSON data has not been loaded yet. Call load() or from_dict() first."
            )
        return self._data

    @data.setter
    def data(self, data):
        self._data = data
        self._validate_data()

    @property
    def entries(self) -> List[Dict[str, Any]]:
        """
        Direct access to 'search-results.entry' (list).
        Return empty list if the contract is fulfilled but there are no results.
        """

        return self._extract_entries()
    
    def _extract_entries(self) -> List[Dict[str, Any]]:
        """
        Core logic: validates and extracts 'search-results.entry'.
        Returns an empty list if it's None or missing.
        """
        if self._data is None:
            raise AttributeError("No data loaded to validate or extract.")

        sr = self._data.get("search-results")
        if not isinstance(sr, dict):
            raise ElsevierSchemaError(
                "Response does not contain 'search-results' as a dict."
            )

        entry = sr.get("entry", [])
        if entry is None:
            return []
        if not isinstance(entry, list):
            raise ElsevierSchemaError(
                "'search-results.entry' must be a list."
            )
        return entry

    def load(self) -> None:
        """
        Reads the JSON file from the local path (used only when the file 
        is stored locally instead of in a database).
        """
        path = self.path
        logger.info("Reading Elsevier's JSON from %s", path)

        with path.open('r', encoding = 'utf-8') as f:
            self.data = json.load(f)

    def from_dict(self, payload: Dict[str, Any]) -> None:
        """
        Directly injects a JSON dict (e.g. request response).
        """
        self.data = payload

    def _validate_data(self) -> None:
        """
        Lightweight validation based on the API contract:
        - 'search-results' must exist and be a dict.
        - 'entry' must exist (list, possibly empty) or be coercible to one.
        """

        _ = self._extract_entries()

        if _ is None:
            logger.warning(
                "'search-results.entry' arrived as None. It will be treated as an empty list"
            )

    @staticmethod
    def preprocess_author(authors):
        return ';'.join(f'{auth["authname"]}' for auth in authors)

    @staticmethod
    def preprocess_author_with_id(authors):
        return ';'.join(f'{auth["authname"]} ({auth["authid"]})' for auth in authors)
    
    @staticmethod
    def preprocess_authorid(authors):
        return ';'.join(f'{auth["authid"]}' for auth in authors)

    def _enrich_entries(self, entries: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        enriched: List[Dict[str, Any]] = []

        for e in entries: 
            authors = e.get("author") or []
            copy_e = dict(e)

            copy_e['authors_id'] = self.preprocess_authorid(authors)
            copy_e['authors_info'] = self.preprocess_author(authors)
            copy_e['authors_info_with_id'] = self.preprocess_author_with_id(authors)
            enriched.append(copy_e)

        return enriched

    def to_dataframe(self) -> pd.DataFrame:
        """
        Normalise 'entries' to DataFrame, apply enrichments and renames.
        Returns empty DataFrame (with expected columns if there are fields_map) when there are no results.
        """

        entries = self.entries
        if not entries: 
            logger.info("Elsevier response with no results: 0 entries.")
            if fields:
                return pd.DataFrame(columns=list(fields.values()))
            return pd.DataFrame()
        
        logger.info("Normalising %d entries from Elsevier.", len(entries))
        enriched = self._enrich_entries(entries)
        df = pd.json_normalize(enriched)

        if fields:
            original_cols = set(df.columns)
            wanted = set(fields.keys())
            not_mapped = original_cols - wanted
            missing_in_df = wanted - original_cols

            if not_mapped:
                logger.info(
                    "Columns present and not mapped (to be ignored): %s",
                    ", ".join(sorted(map(str, not_mapped)))
                )
            if missing_in_df:
                logger.info(
                    "Expected columns in fields_map that did NOT arrive: %s",
                    ", ".join(sorted(map(str, missing_in_df))) 
                )

        keep_cols = [c for c in df.columns if c in fields]
        df = df[keep_cols].rename(columns=fields)

        return df

    def execute(self) -> pd.DataFrame:
        """
        Run the entire pipeline assuming local file input:
        load() → validate contract → enrich → normalise → rename/filter → DataFrame.
        """
        if self._data is None:
            self.load()
        return self.to_dataframe()

In [50]:
prueba = Elsjson(PATH)

# prueba._enrich_entries(prueba.entries)
prueba.execute().head(1)

INFO:__main__:Reading Elsevier's JSON from scopus_results_fieldv2.json
INFO:__main__:Normalising 25 entries from Elsevier.
INFO:__main__:Columns present and not mapped (to be ignored): @_fa, author, prism:pageRange, prism:publicationName, prism:url, subtype


,Title,Volume,Date,DOI,Cited by,Document Type,Author(s) ID,Authors,Author full names,Issue
0,Biogeochemical study of the periglacial slopes...,443,2026-01-01,10.1016/j.icarus.2025.116783,0,Article,58310191500;58310000000;7003652519;60054445900...,Leal M.A.;Tovar D.;de Pablo M.A.;Bonilla M.A.;...,Leal M.A. (58310191500);Tovar D. (58310000000)...,NaN
